# DMapS: inter-core breakdown on the original circuits

The **original** circuits are in `data/in/circuits`. The padded circuits in `data/in/circuits_alt` are **not** analysed here,
because DMapS cannot run them (see [Why there is no padded comparison](#Why-there-is-no-padded-comparison) at the end).

**Result file.** `dmaps_results.json` is a run of `experiments/run.py` over all 24 original circuits on devices A–F, with 5 seeds
per circuit. Devices are defined in `experiments/devices/*.json`. DMapS runs CPA mapping followed by CAR routing
(`MultiModeRouting.cpa_map_car_rout`).

**Metrics.** These follow the DMapS paper (Sec. IV, *Metrics*). They are counted from the routed circuit:
- **Remote SWAP** = SWAP gates between qubits on different chips (`best.remote_swaps`)
- **Telegate** = CX gates between qubits on different chips, i.e. remote CNOTs (`best.telegates`)
- **Inter-core** = Remote SWAP + Telegate (`best.inter_core`)
- **EPR cost** = Remote SWAP × 2 + Telegate × 1 (`best.epr_cost`), the paper's EPR costs per remote operation
- **SWAP** = intra-chip SWAPs (`best.swaps`)

Each circuit uses its best seed: the lowest EPR cost, then the fewest SWAPs, out of 5 seeds, as `run.py` picks it.
A **Gmean** row covers one circuit group; the final row covers the whole suite (24 circuits).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML

HERE = Path.cwd()
IN_DIR = HERE.parent / "in"
RESULTS = "dmaps_results.json"
METRICS = ["remote_swaps", "telegates", "inter_core", "epr_cost", "swaps"]
GROUP_ORDER = ["qasm_25", "qasm_64", "qasm_80", "qasm_95", "qasm_200", "qasm_360"]
DEVICE = {   # from experiments/devices/*.json
    "qasm_25":  "25 qubits, device A: four 4×4 cores, ring",
    "qasm_64":  "64 qubits, device B: four 5×5 cores, ring",
    "qasm_80":  "80 qubits, device C: four 6×5 cores, ring",
    "qasm_95":  "95 qubits, device D: six 5×5 cores, 2×3 core grid",
    "qasm_200": "200 qubits, device E: eight 7×5 cores, 2×4 core grid",
    "qasm_360": "360 qubits, device F: twelve 7×6 cores, 3×4 core grid",
}
NAMES = {"ae": "AE", "ghz": "GHZ", "graphstate": "Graphstate", "qft": "QFT", "qnn": "QNN",
         "random": "Random", "qpeexact": "QPEexact", "su2random": "SU2Random"}

In [2]:
data = json.loads((HERE / RESULTS).read_text())
cfg = data["config"]
print(f"{data['method']}, {data['num_seeds']} seeds from {data['start_seed']}; "
      f"remote SWAP cost = {cfg['remote_swap_cost']}, telegate cost = {cfg['telegate_cost']}")

rows = []
for batch in data["batches"]:
    for r in batch["results"]:
        best = r["best"] or {}
        rows.append({"group": batch["batch"], "circuit": r["file"].split("_nativegates")[0], "file": r["file"],
                     **{m: best.get(m, np.nan) for m in METRICS}})

df = pd.DataFrame(rows)
missing = df[df[METRICS].isna().any(axis=1)]
print(f"{len(df)} results loaded; {len(missing)} circuit(s) with no valid run")
if len(missing):
    display(missing)

# Sanity check: the stored totals match their definitions.
assert (df.inter_core == df.remote_swaps + df.telegates).all()
assert (df.epr_cost == cfg["remote_swap_cost"] * df.remote_swaps + cfg["telegate_cost"] * df.telegates).all()
print("inter_core and epr_cost match remote_swaps and telegates for every circuit")

DMapS cpa_map_car_rout, 5 seeds from 0; remote SWAP cost = 2, telegate cost = 1
24 results loaded; 0 circuit(s) with no valid run
inter_core and epr_cost match remote_swaps and telegates for every circuit


## CX counts

**CX** is the number of `cx` lines in the circuit's `.qasm` file. `cx` is the only two-qubit gate in these circuits.

In [3]:
def cx_count(path):
    return sum(1 for line in open(path) if line.startswith("cx "))


cx = {(g, c): cx_count(IN_DIR / "circuits" / g / f) for g, c, f in df[["group", "circuit", "file"]].itertuples(index=False)}
print(f"CX counted for {len(cx)} circuits")

CX counted for 24 circuits


## Table builder

`table_html` lays the results out like the paper's Table III, one row per circuit. `geomean` needs strictly positive values.
Some circuits have no remote SWAPs or no telegates (e.g. GHZ, Graphstate), so a Gmean over a column that contains a zero
is shown as — instead of silently dropping the zero. Inter-core, EPR cost and SWAP are positive for every circuit.

In [4]:
def geomean(s):
    s = pd.Series(s).dropna()
    if not len(s) or (s <= 0).any():
        return None
    return float(np.exp(np.log(s).mean()))


def gmean_row(label, sub):
    return dict(name=label, cx=None, **{m: geomean(sub[m]) for m in METRICS})


def table_rows(p):
    rows = []
    for g in GROUP_ORDER:
        if g not in p.index.get_level_values("group"):
            continue
        sub = p.xs(g, level="group")
        rows.append(("section", DEVICE[g]))
        for c, r in sub.iterrows():
            rows.append(("circuit", dict(name=NAMES.get(c, c), cx=cx[(g, c)], **{m: r[m] for m in METRICS})))
        rows.append(("gmean", gmean_row(f"Gmean ({len(sub)})", sub)))
    rows.append(("suite", gmean_row(f"Gmean, suite ({len(p)})", p)))
    return rows


CSS = """<style>
.ptab-wrap{display:inline-block;background:#ffffff;padding:18px 24px;border-radius:4px;margin:8px 0 24px}
.ptab-wrap,.ptab-wrap *{color:#0b0b0b !important}
.ptab,.ptab tr,.ptab th,.ptab td,.ptab caption{background:#ffffff !important}
.ptab{font-family:'Times New Roman',Times,serif;font-size:13px;border-collapse:collapse;margin:0}
.ptab caption{caption-side:top;font-variant:small-caps;text-align:center;padding-bottom:8px;max-width:640px;line-height:1.35}
.ptab caption b{display:block;font-variant:normal;letter-spacing:.06em;font-weight:normal}
.ptab th,.ptab td{padding:2px 10px;text-align:right;white-space:nowrap}
.ptab th:first-child,.ptab td:first-child{text-align:left}
.ptab thead tr:first-child th{border-top:1.6px solid #0b0b0b}
.ptab thead tr:last-child th{border-bottom:1px solid #0b0b0b}
.ptab .grp th{font-weight:normal;font-variant:small-caps;border-bottom:.6px solid #0b0b0b;text-align:center}
.ptab .sec td{font-style:italic;text-align:left;padding-top:8px}
.ptab .gm td{border-top:.6px solid #0b0b0b;font-weight:600}
.ptab .suite td{border-top:1px solid #0b0b0b;border-bottom:1.6px solid #0b0b0b;font-weight:600}
</style>"""


def _n(v, gm):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else (f"{v:,.1f}" if gm else f"{int(v):,}")


def table_html(p, number, caption):
    head = ("<thead><tr class='grp'><th></th><th></th><th colspan=3>Inter-core</th><th></th><th></th></tr>"
            "<tr><th>Circuit</th><th>CX</th><th>Remote SWAP</th><th>Telegate</th><th>Total</th>"
            "<th>EPR cost</th><th>SWAP</th></tr></thead>")
    body = []
    for kind, r in table_rows(p):
        if kind == "section":
            body.append(f"<tr class='sec'><td colspan=7>{r}</td></tr>")
            continue
        gm = kind != "circuit"
        cls = {"circuit": "", "gmean": "gm", "suite": "suite"}[kind]
        cells = "".join(f"<td>{_n(r[m], gm)}</td>" for m in METRICS)
        body.append(f"<tr class='{cls}'><td>{r['name']}</td><td>{_n(r['cx'], False)}</td>{cells}</tr>")
    cap = f"<caption><b>TABLE {number}</b>{caption}</caption>"
    return HTML(CSS + f"<div class='ptab-wrap'><table class='ptab'>{cap}{head}<tbody>{''.join(body)}</tbody></table></div>")

## Table I: inter-core breakdown, original circuits

Inter-core operations split into remote SWAPs and telegates, with the EPR cost they imply and the intra-chip SWAPs.

In [5]:
table_html(df.set_index(["group", "circuit"]), "I",
           "DMapS on the original circuits. Inter-core = remote SWAP + telegate; "
           f"EPR cost = {cfg['remote_swap_cost']} × remote SWAP + {cfg['telegate_cost']} × telegate (DMapS paper metric); "
           "SWAP counts intra-chip SWAPs; each circuit uses its best of 5 seeds. "
           "— marks a Gmean over a column that contains a zero.")

## Why there is no padded comparison

The padded circuits in `data/in/circuits_alt` contain exactly the same gates as the originals. The only difference is the
register size: they declare extra qubits that no gate touches (for example, the `qasm_25` circuits declare `qreg q[41]` but use
only 25 qubits). DMapS cannot map these circuits, so `run.py` skips `circuits_alt` (it is commented out of `CIRCUIT_SETS`).

**Where it fails.** DMapS partitions the circuit across chips with a hypergraph that has one vertex per qubit that takes part in a
two-qubit gate (`QuCircPartitioner._build_circ_hyperG`). Qubits that only see single-qubit gates are placed afterwards, into chips
with spare room (`TwoPhaseMapper._adjust_par_res`). Qubits that see **no gates at all** are handled by neither step:
`TwoPhaseMapper._count_used_qubits` only collects qubits that appear in some instruction, so they are never counted or placed.
When the partitioner then builds its initial partition, it loops over *every* qubit of the circuit
(`src/partitioner/qucirc_partitioner.py:104`) and looks each one up in the hypergraph's index. The first idle qubit is not there, so
every padded circuit stops with an error such as

```
KeyError: Qubit(QuantumRegister(41, 'q'), 25)
```

**Why it was not patched.** Running the padded circuits would need a change to DMapS itself: the idle qubits would have to be
counted when DMapS chooses which chips to use, and then given physical qubits. That would no longer be DMapS as published,
so its results would not be a fair DMapS baseline. A patch along these lines was tried and then reverted. The analysis therefore
reports DMapS only on the original circuits.